# Phase 2 — Pattern Extraction (color / stripe / spot)

Runs `src/pattern_extractor/` against Phase 1's output (`data/extracted_fish/`) to produce one feature row per image across three independent dimensions: coloring (patternize-derived k-means clustering), spots (blob shape), stripes (region elongation + FFT periodicity).

**No GPU needed.** This is pure NumPy/SciPy/Pillow, deterministic and fast - CLAUDE.md notes this stage deliberately skips resumable per-image state for exactly that reason, a re-run just recomputes everything, cheaply. Use a **CPU runtime** here (`Runtime -> Change runtime type -> CPU`) to save your GPU quota for Phase 1.

**Prerequisite:** Phase 1 must already have produced output in `data/extracted_fish/` - run `Phase1_Fish_Extraction.ipynb` first.

See [README.md](../README.md) (Planned Approach, step 2) for the full citation and design reasoning (the *patternize* reimplementation, the three-dimension split's developmental-biology basis).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1 used

This must resolve to the **same** `PROJECT_DIR` Phase 1 wrote `data/extracted_fish/` into - Phase 2 reads that output directly via the same relative-path layout. If this is a fresh Colab runtime that never ran Phase 1, this cell clones fresh onto Drive the same way Phase 1's notebook does; if Phase 1 already set it up, this just pulls any code updates.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Add the local package to the path

Base dependencies only (numpy, scipy, Pillow) - already in Colab's default image, no GPU extras needed for this phase.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy, scipy, PIL
print("Base deps OK - numpy", numpy.__version__, "scipy", scipy.__version__, "Pillow", PIL.__version__)

## 4. Confirm Phase 1's output is actually there

In [ ]:
extracted_root = PROJECT_DIR / "data" / "extracted_fish"
species_dirs = list(extracted_root.glob("*/*")) if extracted_root.exists() else []
print(f"{extracted_root}: {len(species_dirs)} species folder(s) found.")
if not species_dirs:
    print("Nothing here yet - run Phase1_Fish_Extraction.ipynb first.")

## 5. Run pattern extraction

Writes one feature row per image (color/spot/stripe columns, plus an `is_reference` column marking the curated seed photo vs. GBIF field images - see README.md's Planned Approach step 3 for why that distinction matters for the aggregation step that comes next in Phase 3) to `reports/pattern_features.csv`.

In [ ]:
from pattern_extractor.config import PipelineConfig
from pattern_extractor.pipeline import PatternExtractorPipeline

config = PipelineConfig()  # relative paths, same PROJECT_DIR layout Phase 1 used
rows = PatternExtractorPipeline(config).run()
print(f"Wrote {len(rows)} feature row(s) to {config.output_csv_path.resolve()}")

## 6. Quick sanity check

In [ ]:
import csv

with open(config.output_csv_path, newline="", encoding="utf-8") as f:
    reader = list(csv.DictReader(f))

n_species_seen = len({row["image_key"].split("/")[1] for row in reader})
n_reference = sum(1 for row in reader if row["is_reference"] == "True")
print(f"{len(reader)} image row(s) across {n_species_seen} species; {n_reference} marked is_reference.")
reader[:3]

## Next: Phase 3

`reports/pattern_features.csv` is Phase 3's input (per-species aggregation + distance matrices - see README.md's Planned Approach, step 3). It's already saved under Drive; pull it down locally, or keep working from Drive, to continue there.